# Biodiversidad vegetal de Perú — panorama con datos de GBIF

Análisis exploratorio de 1,842,228 registros de plantas (Tracheophyta/Plantae) de Perú, obtenidos de la API pública de GBIF (Global Biodiversity Information Facility): evolución en el tiempo, familias botánicas dominantes y regiones con más actividad de registro.

**Nota sobre la fuente:** los datos se obtuvieron mediante consultas agregadas (facets) a la API de GBIF (`https://api.gbif.org/v1/occurrence/search`), guardadas en `data/raw/`. No se descargaron los registros individuales (serían más de 1.8 millones), sino los conteos ya agregados por año, familia y región.

## 1. Carga de los datos crudos (tal como los devuelve la API de GBIF)

In [1]:
import json, unicodedata
import pandas as pd

year_raw = json.load(open("../data/raw/year_counts.json"))["facets"][0]["counts"]
family_raw = json.load(open("../data/raw/family_counts.json"))
region_raw = json.load(open("../data/raw/region_counts_raw.json"))

print(f"Años: {len(year_raw)} | Familias: {len(family_raw)} | Filas de región (con duplicados): {len(region_raw)}")

Años: 100 | Familias: 20 | Filas de región (con duplicados): 30


## 2. Limpieza

- **Años:** se ordenan cronológicamente.
- **Familias:** nos quedamos con las 10 que sí se identificaron por nombre (ver sección de limitaciones: las otras 10 de la consulta original quedaron solo como código numérico, sin resolver).
- **Regiones:** GBIF trae nombres duplicados por tildes y variantes de escritura (`Cusco`/`Cuzco`, `San Martín`/`San Martin`, `Junín`/`Junin`, `Apurimac`/`Apurímac`, y `Madre de Dios` repetido). Se normalizan y se suman.

In [2]:
df_year = pd.DataFrame(year_raw).rename(columns={"name": "anio", "count": "registros"})
df_year["anio"] = df_year["anio"].astype(int)
df_year = df_year.sort_values("anio").reset_index(drop=True)

df_family = pd.DataFrame(family_raw)
df_family_top10 = (df_family[~df_family["name"].str.startswith("Unresolved")]
                    .sort_values("count", ascending=False)
                    .rename(columns={"name": "familia", "count": "registros"})[["familia","registros"]])

def normaliza(s):
    s = unicodedata.normalize("NFKD", s).encode("ascii", "ignore").decode("utf-8").strip().lower()
    return "cusco" if s == "cuzco" else s

df_region = pd.DataFrame(region_raw)
df_region["clave"] = df_region["name"].apply(normaliza)
duplicadas = df_region.groupby("clave").size()
print("Regiones que aparecían más de una vez:\n", duplicadas[duplicadas > 1])

preferido = df_region.sort_values("count", ascending=False).groupby("clave")["name"].first()
region_clean = (df_region.groupby("clave")["count"].sum().reset_index()
                .merge(preferido.rename("nombre"), on="clave")
                .sort_values("count", ascending=False)[["nombre","count"]]
                .rename(columns={"count":"registros"}).reset_index(drop=True))
region_clean.head(10)

Regiones que aparecían más de una vez:
 clave
apurimac         2
cusco            2
junin            2
madre de dios    2
san martin       2
dtype: int64


,nombre,registros
0,Loreto,472779
1,Pasco,226678
2,Cusco,157445
3,Madre de Dios,106091
4,Amazonas,75413
5,Cajamarca,73439
6,San Martín,69369
7,Junín,43999
8,Huánuco,38096
9,La Libertad,20380


## 3. Validaciones

In [3]:
total_api = 1_842_228  # campo "count" de la respuesta de la API (el total real, sin filtrar por facetLimit)
assert df_year.registros.sum() <= total_api
assert df_family_top10.registros.sum() <= total_api
assert region_clean.registros.sum() <= total_api

print(f"Total real (API):           {total_api:,}")
print(f"Suma años 1778-2026 (top100 años): {df_year.registros.sum():,}  "
      f"({df_year.registros.sum()/total_api:.1%} del total)")
print(f"Suma top 10 familias:        {df_family_top10.registros.sum():,}  "
      f"({df_family_top10.registros.sum()/total_api:.1%} del total)")
print(f"Suma regiones (25, limpias): {region_clean.registros.sum():,}  "
      f"({region_clean.registros.sum()/total_api:.1%} del total)")

Total real (API):           1,842,228
Suma años 1778-2026 (top100 años): 1,600,173  (86.9% del total)
Suma top 10 familias:        925,332  (50.2% del total)
Suma regiones (25, limpias): 1,428,120  (77.5% del total)


## 4. Gráficas

### 4.1 Registros por año

In [4]:
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

GREEN, OCHRE, MOSS = "#1F5C3A", "#A97327", "#6B8F71"

fig, ax = plt.subplots(figsize=(9, 3.6), dpi=170)
d = df_year[df_year.anio >= 1950]
ax.bar(d.anio, d.registros, color=MOSS, width=0.8)
ax.set_title("Registros de plantas de Perú en GBIF por año (1950-2026)", fontsize=10)
ax.set_ylabel("N.º de registros")
for s in ["top", "right"]: ax.spines[s].set_visible(False)
plt.tight_layout(); plt.savefig("../images/01_registros_por_anio.png"); plt.show()

### 4.2 Top 10 familias botánicas

In [5]:
fig, ax = plt.subplots(figsize=(8, 4.2), dpi=170)
d = df_family_top10.sort_values("registros")
ax.barh(d.familia, d.registros, color=GREEN)
ax.set_title("Las 10 familias de plantas más registradas en Perú (GBIF)", fontsize=10)
ax.set_xlabel("N.º de registros")
for s in ["top", "right"]: ax.spines[s].set_visible(False)
plt.tight_layout(); plt.savefig("../images/02_top_familias.png"); plt.show()

### 4.3 Top 10 regiones

In [6]:
fig, ax = plt.subplots(figsize=(8, 4.2), dpi=170)
d = region_clean.sort_values("registros", ascending=False).head(10).sort_values("registros")
ax.barh(d.nombre, d.registros, color=OCHRE)
ax.set_title("Las 10 regiones de Perú con más registros botánicos (GBIF)", fontsize=10)
ax.set_xlabel("N.º de registros")
for s in ["top", "right"]: ax.spines[s].set_visible(False)
plt.tight_layout(); plt.savefig("../images/03_top_regiones.png"); plt.show()

## 5. Exportar datos limpios para SQL y el dashboard

In [7]:
df_year.to_csv("../data/year_clean.csv", index=False)
df_family_top10.to_csv("../data/family_clean.csv", index=False)
region_clean.to_csv("../data/region_clean.csv", index=False)
print("Listo.")

Listo.


## 6. Conclusión

- Más de la mitad de los 1.8 millones de registros se concentra en una sola década (2000-2009), con un pico extremo en 2006-2008 — probablemente un proyecto puntual de digitalización de herbarios, no un salto real en la exploración de campo (pendiente de confirmar).
- **Arecaceae** (las palmeras) domina con el 18.6% de todos los registros — casi el triple que la segunda familia.
- **Loreto y Pasco** concentran la mayor actividad de registro regional. La importancia de Pasco amerita revisarse más adelante.
- Ver limitaciones completas en el README del repositorio.